# 容器查询

学习目标：能让同一卡片按祖先容器的可用尺寸调整布局，并说明容器选择、单位与样式查询的边界。

前置知识：HTML 祖先与后代、CSS 层叠、Grid、媒体查询、盒模型和基本相对单位。

适用范围：CSS Conditional Rules Level 5 中的尺寸容器查询及容器单位；样式查询属于持续演进的内容，只示范自定义属性条件，并保留不依赖查询的基础样式。 配套页面不依赖 JavaScript。

环境准备：[环境配置与运行](README.md)。

配套脚本：位于 scripts/14-container-queries/。

1. [index.html](scripts/14-container-queries/index.html)：卡片与容器类型。
2. [nested.html](scripts/14-container-queries/nested.html)：祖先搜索与嵌套条件。
3. [units.html](scripts/14-container-queries/units.html)：容器单位与回退。
4. [style-query.html](scripts/14-container-queries/style-query.html)：样式查询与显式类回退。
5. [styles.css](scripts/14-container-queries/styles.css)：本章各页面的实验规则及少量阅读辅助样式。

Step 1：在已激活 Python 环境的终端中，从项目根目录进入本技术目录。

```bash
cd content/Web与应用开发/css
```

Step 2：启动本章预览服务。

```bash
python -m http.server 8101 --bind 127.0.0.1
```

Step 3：打开[本章示例首页](http://127.0.0.1:8101/scripts/14-container-queries/index.html)。

服务根目录为 content/Web与应用开发/css；保存修改后刷新页面

Step 4：在服务终端按 Ctrl+C 停止服务。

本章使用的主要属性：

| 完整属性名 | 中文名称／含义 | 用途或作用对象 |
| --- | --- | --- |
| container-type | 查询容器类型 | 祖先元素启用所需的尺寸查询轴 |
| container-name | 查询容器名称 | 筛选可查询祖先 |
| container | 容器简写 | 同时设置名称与类型 |
| inline-size | 行内轴尺寸 | 设置容器可用空间 |
| block-size | 块轴尺寸 | 为双轴尺寸隔离提供尺寸 |
| grid-template-columns | 显式列轨道 | 卡片单列与双列布局 |
| gap | 行列间距 | 卡片图文间隔 |
| font-size | 字号 | 容器单位示例 |
| border | 边框简写 | 显示查询条件是否改变后代 |
| --density | 本例自定义的密度属性 | 供样式查询读取；不是内置属性 |

## 1 同一卡片根据所在空间调整

媒体查询观察视口等环境条件；尺寸容器查询观察页面内选中的祖先盒子。两张卡片即使共用一个视口，也可能因各自可用空间不同而采用不同布局。

先让包裹卡片的元素成为查询容器，再在 @container 中修改卡片后代。这里 card-space 是容器名称，.card-host 是类选择器，二者不会自动关联。

container: card-space / inline-size 是简写：斜杠前是 container-name，后是 container-type。inline-size 启用容器自身行内轴的尺寸查询；本例横排时对应宽度。基础单列写在查询外，不支持尺寸查询时仍可阅读。

```html
<div class="card-host wide-host">
  <article class="responsive-card"><div class="card-picture">封面</div><div><h3>布局课程</h3><p>同一结构放在宽容器。</p></div></article>
</div>
<div class="card-host narrow-host">
  <article class="responsive-card"><div class="card-picture">封面</div><div><h3>布局课程</h3><p>同一结构放在窄容器。</p></div></article>
</div>
```

```css
.card-host {
  container: card-space / inline-size;
  margin-bottom: 16px;
}
.wide-host { inline-size: min(700px, 100%); }
.narrow-host { inline-size: min(280px, 100%); }
.responsive-card {
  display: grid;
  grid-template-columns: minmax(0, 1fr);
  gap: 16px;
  border: 1px solid teal;
  padding: 12px;
}
.responsive-card > * { min-width: 0; overflow-wrap: anywhere; }
.card-picture { background-color: #dfefea; padding: 16px; }
@container card-space (min-width: 480px) {
  .responsive-card {
    grid-template-columns: 120px minmax(0, 1fr);
    /* 同一宽视口下：宽容器图文并排，窄容器仍上下排列。 */
  }
}
```

配套文件：[index.html](scripts/14-container-queries/index.html)、[styles.css](scripts/14-container-queries/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/14-container-queries/index.html)

## 2 尺寸隔离与可以查询的轴

container-type 的默认值 normal 不建立尺寸查询容器。inline-size 在行内轴隔离内容对尺寸的影响；size 在行内轴和块轴都进行尺寸隔离。它们还带来相关的样式隔离与独立格式化上下文，不能当作毫无布局影响的标记。

隔离是为了避免“后代变大→容器变大→条件翻转→后代变小”的尺寸循环。容器尺寸需要由父布局或明确的尺寸约束给出；横排块盒通常能从外部获得宽度，但 size 容器不能再指望子内容把自动高度撑起来。

width、height 查询的是容器内容盒。inline-size、block-size 查询逻辑方向的内容盒尺寸；查询涉及的轴必须可用。横排 inline-size 容器无法单凭自身提供 height 查询；若更外层有符合条件的双轴容器，搜索可能选择那里。

本例 size 容器明确给出 block-size，并把长内容设为局部滚动，避免内容数量变化时被截断。

```html
<div class="two-axis-box"><p class="height-marker">此处同时提供宽与高。</p></div>
```

```css
.two-axis-box {
  box-sizing: border-box;
  container-name: panel-size;
  container-type: size;
  inline-size: min(420px, 100%);
  block-size: 180px;
  border: 2px solid gray;
  overflow: auto;
}
.height-marker { margin: 8px; padding: 8px; border: 3px solid gray; }
@container panel-size (height >= 160px) {
  .height-marker { border-color: teal; }
  /* 180px 为边框盒高度，扣除两边边框后仍满足 160px 的内容盒条件。 */
}
```

配套文件：[index.html](scripts/14-container-queries/index.html)、[styles.css](scripts/14-container-queries/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/14-container-queries/index.html)

## 3 容器不能用自身尺寸查询自身

对规则中被选择的普通元素，浏览器从它的祖先中选择查询容器，不把该元素自身列入候选。把 container-type 和 @container 中的目标类写在同一个盒子上，不会让它观察自己的尺寸。

这不代表一个容器永远不能被条件样式改变：它可以是另一祖先容器的后代，从祖先条件获得样式。本例特意不给 self-box 更外层同名容器，让自身和直接子段落形成对照；不涉及伪元素的特殊情况。

```html
<div class="self-box">
  <p class="self-child">后代可以查询这个盒子。</p>
</div>
```

```css
.self-box {
  container: solo / inline-size;
  inline-size: min(500px, 100%);
  border: 4px solid gray;
}
.self-child { margin: 8px; color: black; }
@container solo (min-width: 300px) {
  .self-box { border-color: red; }
  .self-child { color: teal; }
  /* 宽容器下：外框仍灰色，子段落变蓝绿色；外框不查询自身。 */
}
```

配套文件：[nested.html](scripts/14-container-queries/nested.html)、[styles.css](scripts/14-container-queries/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/14-container-queries/nested.html)

## 4 命名筛选与最近合格祖先

container-name 默认是 none，表示没有名字，不表示“任何查询都禁止”。不带名称的尺寸查询会找最近能提供所需轴的祖先；有名字的查询再增加名称筛选。名称区分大小写，可以给一个容器多个名字，名字不需要在文档里唯一。

选定最近合格容器之后才判断尺寸阈值。若近处容器类型和名字都符合、但宽度不满足条件，不会继续向外找一个宽度更大的同名容器。名称筛选也不是 CSS 选择器：不能写成 .zone 或 #zone。

外层同时叫 zone 和 outer-zone，内层只叫 zone。.nearest-marker 由内层决定，.named-marker 用 outer-zone 跳过内层。

```html
<div class="outer-scope">
  <p class="outer-marker">外层的直接后代</p>
  <div class="inner-scope">
    <p class="nearest-marker">最近同名容器</p>
    <p class="named-marker">指定外层名称</p>
    <p class="both-conditions">两个容器的条件同时成立</p>
  </div>
</div>
```

```css
.outer-scope {
  container-name: zone outer-zone;
  container-type: inline-size;
  inline-size: min(700px, 100%);
  border: 1px solid gray;
}
.inner-scope {
  container: zone / inline-size;
  inline-size: min(260px, 100%);
  border: 1px solid gray;
}
.outer-scope p { padding: 4px; }
.outer-marker, .nearest-marker, .named-marker { color: black; }
@container zone (min-width: 500px) {
  .outer-marker, .nearest-marker { color: teal; }
}
@container outer-zone (min-width: 500px) {
  .named-marker { color: teal; }
  /* 外层足够宽时，nearest-marker 仍黑色，named-marker 为蓝绿色。 */
}
```

配套文件：[nested.html](scripts/14-container-queries/nested.html)、[styles.css](scripts/14-container-queries/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/14-container-queries/nested.html)

## 5 嵌套条件可以来自不同容器

嵌套 @container 要求外层和内层条件都成立，但每一层可以选择不同的祖先。本节复用上一节的结构；两个名称分别选到外层和内层盒子。

不要机械合并为单个“宽度至少 500px 且至多 300px”的查询，那样会要求同一个容器同时满足矛盾条件。容器查询条件中的 em 等单位还会按被查询容器的字号解释，和媒体查询使用初始字体值的规则不同。

```html
<!-- 与上一组相同的嵌套结构；该独立组用于只观察两层条件。 -->
<div class="outer-scope">
  <div class="inner-scope">
    <p class="both-conditions">两个容器的条件同时成立</p>
  </div>
</div>
```

```css
.both-conditions { border: 3px solid gray; }
@container outer-zone (min-width: 500px) {
  @container zone (max-width: 300px) {
    .both-conditions {
      border-style: dashed;
      /* 外层至少 500px、内层至多 300px 时出现虚线。 */
    }
  }
}
```

配套文件：[nested.html](scripts/14-container-queries/nested.html)、[styles.css](scripts/14-container-queries/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/14-container-queries/nested.html)

## 6 容器单位及无容器时的回退

- cqw、cqh 分别为可查询容器宽、高的 1%；cqi、cqb 分别为行内、块方向尺寸的 1%。
- cqmin、cqmax 分别取 cqi 与 cqb 中较小、较大的单位长度。两轴可能从不同合格祖先取得，不能默认始终来自同一个盒子。
- 单位选择相关轴上最近的合格祖先。@container 指定的名字不会把内部所有容器单位永久绑定到该命名容器。

没有合格容器时，相应单位按小视口尺寸（small viewport）计算，不会自动变成 0 或让前一条声明回退。为了保证组件字号以容器为参照，可把单位放在确认该轴可查询的条件内，并在条件外保留 rem 基础值。

clamp() 为字号保留上下界。下方单独的 no-container-unit 故意没有尺寸容器祖先，用来和同宽视口的容器条比较。

```html
<div class="unit-host"><div class="unit-bar">50cqi</div><p class="unit-text">跟随容器的字号</p></div>
<div class="unit-host unit-wide"><div class="unit-bar">50cqi</div><p class="unit-text">跟随容器的字号</p></div>
<div class="no-container-unit">无容器：10cqi</div>
```

```css
.unit-host {
  container-type: inline-size;
  inline-size: min(240px, 100%);
  margin-bottom: 16px;
  outline: 1px solid gray;
}
.unit-wide { inline-size: min(600px, 100%); }
.unit-bar { inline-size: 50%; background-color: #dfefea; }
.unit-text { font-size: 1rem; }
@container (inline-size >= 0px) {
  .unit-bar { inline-size: 50cqi; }
  .unit-text { font-size: clamp(1rem, 4cqi, 2rem); }
  /* 两条宽度分别为最近容器内容宽度的一半；字号有 rem 上下限。 */
}
.no-container-unit {
  inline-size: 10vw;
  inline-size: 10cqi;
  overflow-wrap: anywhere;
  border: 1px solid teal;
  /* 支持 cqi 但没有容器时，以小视口行内尺寸计算；前一条是语法不支持回退。 */
}
```

配套文件：[units.html](scripts/14-container-queries/units.html)、[styles.css](scripts/14-container-queries/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/14-container-queries/units.html)

## 7 了解自定义属性样式查询

尺寸查询看盒子实际尺寸；style() 样式查询检查容器计算后的样式值。这里 --density 是自定属性，compact 是自定值，不是 container-type 的关键字；自定义属性的继承机制会在“自定义属性与主题”中展开。

普通元素默认可作为样式查询容器，不需设置 container-type: inline-size。container-type: normal 和 container-name: none 都不等于“禁止样式查询”。不带名称的样式查询会考虑最近合格祖先，若自定义属性被继承，中间祖先也可能拥有这个值；命名查询可以明确边界。

本章只示范 style(--density: compact)。不能因为尺寸查询已支持，就承诺任意 CSS 属性、范围式样式查询等分支也可用；须单独核对 @container 的样式查询支持情况。这里使用 CSS Conditional Rules Level 5 中的自定义属性条件，按增强功能处理。

基础列表始终保留完整内容。显式 class 提供同样紧凑间距的回退对照，样式查询组只增强间距和边框；没有支持时保持普通列表。

```html
<div class="density-host">
  <ul class="density-list"><li>第一项</li><li>第二项</li></ul>
</div>
<div class="density-fallback">
  <ul class="density-list"><li>显式类回退：第一项</li><li>第二项</li></ul>
</div>
```

```css
.density-host {
  container-name: density-space;
  --density: compact;
}
.density-list { padding: 16px 32px; border: 2px solid gray; }
.density-list li { margin-block: 12px; }
.density-fallback .density-list { padding-block: 8px; }
.density-fallback .density-list li { margin-block: 4px; }
@container density-space style(--density: compact) {
  .density-list { padding-block: 8px; border-color: teal; }
  .density-list li { margin-block: 4px; }
  /* 支持时与显式类回退一样紧凑；不支持时内容和列表标记仍在。 */
}
```

配套文件：[style-query.html](scripts/14-container-queries/style-query.html)、[styles.css](scripts/14-container-queries/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/14-container-queries/style-query.html)

## 本章小结

- 尺寸查询需要合适的容器类型和外部尺寸来源，查询的是内容盒。
- 普通元素查询祖先，不能依靠自身尺寸查询自身。
- 容器名称筛选候选范围；选定最近合格祖先后才判断阈值，不因阈值为假继续外找。
- 嵌套查询可以分别观察不同容器；容器单位也要逐轴判断参照。
- 样式查询与尺寸查询的默认候选不同，自定义属性条件需独立核对支持并保留基础样式。

## 练习

（1）不修改浏览器视口，只把首页窄容器的 inline-size 改为 520px，并保持 max-width 等价的上限约束。检查卡片是否能切为双列，说明条件参照。

（2）把 nested.html 第一组 inner-scope 的名称从 zone 改为 local。预测 nearest-marker 会选哪个祖先，再查看计算颜色。

（3）在 units.html 给 unit-host 加 20px 内边距，保持 border-box。比较 50cqi 对应内容盒还是边框盒；再移除 container-type，检查条件外的字号回退。

### 提示

宽度是否达到阈值要扣除内边距和边框。修改名称后检查查询的候选集合；不要只盯着条件里的数字。

## 参考与引用来源

- W3C：[box-sizing](https://www.w3.org/TR/css-sizing-3/#box-sizing)：尺寸设置采用的盒模型；[CSS Conditional Rules Level 5 §5.1–5.4](https://www.w3.org/TR/css-conditional-5/#container-queries)、[尺寸特性](https://www.w3.org/TR/css-conditional-5/#size-container)、[样式特性](https://www.w3.org/TR/css-conditional-5/#style-container)、[容器相对长度](https://www.w3.org/TR/css-conditional-5/#container-lengths)：类型、名称、祖先选择、嵌套、内容盒与无合格容器的单位回退。
- MDN：[CSS container queries](https://developer.mozilla.org/en-US/docs/Web/CSS/Guides/Containment/Container_queries)、[Using container size and style queries](https://developer.mozilla.org/en-US/docs/Web/CSS/Guides/Containment/Container_size_and_style_queries)：组件用途、候选范围、自定义属性样式查询及支持限制；[container-type](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/container-type)、[container-name](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/container-name)、[@container](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/At-rules/@container)、[length 的容器单位](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Values/length#container_query_length_units)：尺寸隔离、名称和具体语法。
- Python 3.12：[http.server 命令行](https://docs.python.org/3.12/library/http.server.html#command-line-interface)：服务工作目录、端口和绑定地址。